## DL GENAI Project: Smart MCQ Solver Challenge
 
### **Course:** Deep Learning & Generative AI (Diploma Level)
### **Student:** Ankit Kumar | **Roll No:** 23f3000080
### **Kaggle Notebook:** DL-23f3000080-notebook-t22026
### **W&B Project:** 23f3000080-t22026
### **Target Score:** > 0.73 mAP@3

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


### 1. Setup and Imports

In [2]:
# Install required packages
!pip install -q transformers datasets accelerate scikit-learn wandb

# Imports
import os
import gc
import re
import time
import random
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import wandb
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

set_all_seeds(42)

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

### 2. Configuration

In [3]:
@dataclass
class Config:
    """Configuration class for the project."""
    
    # Data paths
    data_path: str = "/kaggle/input/competitions/smart-mcq-solver-challenge"
    max_length: int = 256
    train_batch_size: int = 16
    eval_batch_size: int = 32
    
    # Training parameters
    epochs: int = 4
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    gradient_accumulation_steps: int = 2
    max_grad_norm: float = 1.0
    early_stopping_patience: int = 2
    
    # Scratch model parameters
    scratch_embedding_dim: int = 512
    scratch_hidden_dim: int = 768
    scratch_num_layers: int = 3
    scratch_num_classes: int = 5
    scratch_max_sequence_length: int = 200
    scratch_dropout: float = 0.3
    
    # WandB
    wandb_project: str = "23f3000080-t22026"
    wandb_api_key: str = "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"
    
    # Option columns
    option_columns: List[str] = field(default_factory=lambda: ['A', 'B', 'C', 'D', 'E'])
    options_mapping: Dict = field(default_factory=lambda: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4})
    idx_to_option: Dict = field(default_factory=lambda: {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'})
    
    # Ensemble weights
    ensemble_weights: List[float] = field(default_factory=lambda: [0.1, 0.45, 0.45])

config = Config()


### 3. Data Preprocessing

In [4]:
class AdvancedDataPreprocessor:
    """Advanced data preprocessing for MCQ answering."""
    
    def __init__(self):
        self.vocab = {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}
        self.word2idx = self.vocab.copy()
        self.idx2word = {v: k for k, v in self.vocab.items()}
        self.vocab_size = len(self.vocab)
        self.max_length = config.scratch_max_sequence_length
        
    def clean_text(self, text):
        """Advanced text cleaning"""
        if pd.isna(text):
            return ''
        text = str(text).lower()
        text = re.sub(r'[^a-zA-Z0-9\s\.\,\?\'\-\:]', ' ', text)
        text = ' '.join(text.split())
        return text
    
    def load_data(self):
        """Load and preprocess the training data"""
        print("Loading data...")
        train_df = pd.read_csv(f'{config.data_path}/train.csv')
        
        print(f"Loaded {len(train_df)} samples")
        print(f"Answer distribution:\n{train_df['answer'].value_counts()}")
        
        # Clean text
        print("\nCleaning text...")
        train_df['clean_prompt'] = train_df['prompt'].apply(self.clean_text)
        
        for col in config.option_columns:
            train_df[f'clean_{col}'] = train_df[col].apply(self.clean_text)
        
        # Create multiple text representations
        train_df['combined_text'] = train_df.apply(
            lambda row: self.create_enhanced_text(
                row['clean_prompt'],
                [row[f'clean_{col}'] for col in config.option_columns]
            ), axis=1
        )
        
        # Clean answers
        train_df['answer'] = train_df['answer'].str.strip()
        
        # Build vocabulary
        print("\nBuilding vocabulary...")
        self.build_vocabulary(train_df['combined_text'].tolist())
        print(f"Vocabulary size: {self.vocab_size}")
        
        # Convert to sequences
        X = self.texts_to_sequences(train_df['combined_text'].tolist())
        y = train_df['answer'].map(config.options_mapping).values
        
        return X, y, train_df
    
    def create_enhanced_text(self, prompt: str, options: List[str]) -> str:
        """Create enhanced text with special formatting"""
        enhanced = f"[CLS] question: {prompt} [SEP] "
        for i, opt in enumerate(options):
            if opt and opt != '':
                enhanced += f"option_{chr(65+i)}: {opt} [SEP] "
        enhanced += "[CLS]"
        return enhanced
    
    def build_vocabulary(self, texts):
        """Build vocabulary with minimum frequency threshold"""
        word_counts = Counter()
        for text in texts:
            words = text.split()
            word_counts.update(words)
        
        min_freq = 2
        vocab_words = [word for word, count in word_counts.items() 
                      if count >= min_freq]
        
        for word in vocab_words[:40000 - len(self.vocab)]:
            if word not in self.word2idx:
                self.word2idx[word] = len(self.word2idx)
                self.idx2word[len(self.idx2word)] = word
        
        self.vocab_size = len(self.word2idx)
    
    def texts_to_sequences(self, texts):
        """Convert texts to padded sequences"""
        sequences = []
        for text in texts:
            words = text.split()
            seq = [self.word2idx.get(word, self.word2idx['<UNK>']) for word in words]
            seq = seq[:self.max_length]
            seq = seq + [self.word2idx['<PAD>']] * (self.max_length - len(seq))
            sequences.append(seq)
        return np.array(sequences, dtype=np.int64)

### 4. Dataset Classes

In [5]:
class MCQDataset(Dataset):
    """Custom Dataset for MCQ Answer Selection."""
    
    def __init__(self, texts, labels, tokenizer, max_length=256, augment=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augment = augment
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Data augmentation
        if self.augment and np.random.random() < 0.15:
            text = self.augment_text(text)
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }
    
    def augment_text(self, text):
        """Text augmentation: random token masking"""
        if isinstance(text, str):
            words = text.split()
            if len(words) > 5:
                mask_idx = np.random.choice(len(words), size=max(1, int(len(words)*0.05)), replace=False)
                for idx in mask_idx:
                    words[idx] = '[MASK]'
                return ' '.join(words)
        return text

class ScratchDataset(Dataset):
    """Dataset for scratch model."""
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.texts[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def prepare_data(df, tokenizer, max_length=256, is_train=True, augment=False):
    """Prepare data for training/inference."""
    texts = []
    labels = []
    
    option_cols = ['A', 'B', 'C', 'D', 'E'] if 'E' in df.columns else ['A', 'B', 'C', 'D']
    
    for idx, row in df.iterrows():
        question = str(row['prompt'])
        
        for opt_idx, col in enumerate(option_cols):
            option = str(row[col])
            text = f"Question: {question} Option: {option}"
            texts.append(text)
            
            if is_train and 'answer' in row:
                answer_idx = ord(row['answer']) - ord('A')
                label = 1 if opt_idx == answer_idx else 0
                labels.append(label)
    
    dataset = MCQDataset(texts, labels, tokenizer, max_length, augment=augment)
    return dataset

### 5. Model 1: Enhanced Scratch Model

In [6]:
class PositionalEncoding(nn.Module):
    """Positional encoding for transformer."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerBlock(nn.Module):
    """Transformer block with attention and feed-forward."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        ff_output = self.ff(x)
        x = self.norm2(x + ff_output)
        return x

class EnhancedScratchModel(nn.Module):
    """Enhanced scratch model with transformer architecture."""
    
    def __init__(self, vocab_size, embedding_dim=512, hidden_dim=768, 
                 num_layers=3, num_heads=8, max_length=200, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.positional = PositionalEncoding(embedding_dim, max_length)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            TransformerBlock(embedding_dim, num_heads, hidden_dim, dropout)
            for _ in range(num_layers)
        ])
        
        # Multi-scale feature extraction
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim * 3, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 5)
        )

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = self.positional(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        # Multi-scale pooling
        mean_pooled = x.mean(dim=1)
        max_pooled, _ = x.max(dim=1)
        weighted_pool = torch.sum(x * F.softmax(x.mean(dim=1, keepdim=True), dim=1), dim=1)
        pooled = torch.cat([mean_pooled, max_pooled, weighted_pool], dim=1)
        
        logits = self.classifier(pooled)
        return logits

### 6. Model 2 & 3: Pretrained Models

In [7]:
class PretrainedMCQModel(nn.Module):
    """Model 2 & 3: Pretrained transformer models."""
    
    def __init__(self, model_name='bert-base-uncased', dropout=0.3):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.num_labels = 2
        
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            config=self.config,
            ignore_mismatched_sizes=True
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        logits = self.dropout(outputs.logits)
        return {'logits': logits, 'loss': outputs.loss if labels is not None else None}

### 7. Training Functions

In [8]:
def train_epoch_scratch(model, train_loader, optimizer, criterion):
    """Train scratch model for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(train_loader), correct / total

def validate_scratch(model, val_loader):
    """Validate scratch model"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model(input_ids)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return accuracy, f1, all_preds, all_labels

def train_scratch_model(X, y, preprocessor):
    """Train scratch model with cross-validation"""
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    fold_results = []
    best_model_state = None
    
    print("\n" + "="*60)
    print("Training Scratch Model with Cross-Validation")
    print("="*60)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'='*50}")
        print(f"Fold {fold + 1}/3")
        print(f"{'='*50}")
        
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]
        
        train_dataset = ScratchDataset(X_train_fold, y_train_fold)
        val_dataset = ScratchDataset(X_val_fold, y_val_fold)
        
        train_loader = DataLoader(train_dataset, batch_size=config.train_batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config.eval_batch_size, shuffle=False)
        
        model = EnhancedScratchModel(
            vocab_size=preprocessor.vocab_size,
            embedding_dim=config.scratch_embedding_dim,
            hidden_dim=config.scratch_hidden_dim,
            num_layers=config.scratch_num_layers,
            max_length=config.scratch_max_sequence_length,
            dropout=config.scratch_dropout
        ).to(DEVICE)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
        best_val_acc = 0
        patience_counter = 0
        
        for epoch in range(15):
            train_loss, train_acc = train_epoch_scratch(model, train_loader, optimizer, criterion)
            val_acc, val_f1, _, _ = validate_scratch(model, val_loader)
            scheduler.step()
            
            print(f"Epoch {epoch+1:2d}/15 | Train Loss: {train_loss:.4f} | "
                  f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
                if patience_counter >= 5:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
        
        fold_results.append(best_val_acc)
        print(f"\nFold {fold + 1} Best Validation Accuracy: {best_val_acc:.4f}")
    
    print(f"\nAverage Cross-Validation Accuracy: {np.mean(fold_results):.4f}")
    
    return best_model_state, np.mean(fold_results)

def train_pretrained_model(model, train_loader, val_loader, config, model_name):
    """Train a pretrained model."""
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    
    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    model.to(DEVICE)
    scaler = GradScaler()
    best_val_acc = 0.0
    best_val_f1 = 0.0
    
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print(f"{'='*60}")
    
    for epoch in range(config.epochs):
        # Training Phase
        model.train()
        train_loss = 0
        train_preds, train_labels = [], []
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Train"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(input_ids, attention_mask, labels)
                loss = outputs['loss'] / config.gradient_accumulation_steps
            
            scaler.scale(loss).backward()
            
            if (len(train_preds) + 1) % config.gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * config.gradient_accumulation_steps
            preds = torch.argmax(outputs['logits'], dim=-1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())
        
        train_acc = accuracy_score(train_labels, train_preds)
        train_f1 = f1_score(train_labels, train_preds, average='weighted')
        
        # Validation Phase
        model.eval()
        val_loss = 0
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Val"):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask, labels)
                val_loss += outputs['loss'].item()
                preds = torch.argmax(outputs['logits'], dim=-1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='weighted')
        
        # Log to WandB
        if wandb.run is not None:
            wandb.log({
                f'{model_name}_train_loss': train_loss/len(train_loader),
                f'{model_name}_train_acc': train_acc,
                f'{model_name}_train_f1': train_f1,
                f'{model_name}_val_loss': val_loss/len(val_loader),
                f'{model_name}_val_acc': val_acc,
                f'{model_name}_val_f1': val_f1,
                'epoch': epoch + 1,
                'learning_rate': scheduler.get_last_lr()[0]
            })
        
        print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f'{model_name}_best.pt')
    
    # Load best model
    try:
        model.load_state_dict(torch.load(f'{model_name}_best.pt'))
    except:
        pass
    
    return model, best_val_acc, best_val_f1

### 8. Setup WandB

In [9]:
def setup_wandb():
    """Setup Weights & Biases for experiment tracking."""
    try:
        if config.wandb_api_key:
            wandb.login(key=config.wandb_api_key)
            wandb.init(
                project=config.wandb_project,
                name=f"mcq_training_{int(time.time())}",
                config=vars(config)
            )
            print(f"WandB initialized: {wandb.run.name}")
            return True
        else:
            print("WandB API key not set. Continuing without WandB...")
            return False
    except Exception as e:
        print(f"WandB setup failed: {e}")
        print("Continuing without WandB...")
        return False

# Setup WandB
wandb_enabled = setup_wandb()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3000080 (23f3000080-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260721_215824-i4kskway
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mcq_training_1784671104
wandb: ⭐️ View project at https://wandb.ai/23f3000080-dl-genai-project/23f3000080-t22026
wandb: 🚀 View run at https://wandb.ai/23f3000080-dl-genai-project/23f3000080-t22026/runs/i4kskway


WandB initialized: mcq_training_1784671104


### 9. Load Data

In [10]:
try:
    train_df = pd.read_csv(f"{config.data_path}/train.csv")
    test_df = pd.read_csv(f"{config.data_path}/test.csv")
    print(f"✅ Loaded {len(train_df)} training samples")
    print(f"✅ Loaded {len(test_df)} test samples")
except:
    print("Creating dummy data...")
    np.random.seed(42)
    n_train, n_test = 2000, 500
    train_df = pd.DataFrame({
        'id': range(n_train),
        'prompt': [f'Question {i}' for i in range(n_train)],
        'A': [f'Option A {i}' for i in range(n_train)],
        'B': [f'Option B {i}' for i in range(n_train)],
        'C': [f'Option C {i}' for i in range(n_train)],
        'D': [f'Option D {i}' for i in range(n_train)],
        'E': [f'Option E {i}' for i in range(n_train)],
        'answer': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_train)
    })
    test_df = pd.DataFrame({
        'id': range(n_test),
        'prompt': [f'Test Q {i}' for i in range(n_test)],
        'A': [f'Opt A {i}' for i in range(n_test)],
        'B': [f'Opt B {i}' for i in range(n_test)],
        'C': [f'Opt C {i}' for i in range(n_test)],
        'D': [f'Opt D {i}' for i in range(n_test)],
        'E': [f'Opt E {i}' for i in range(n_test)]
    })

print(f"\nAnswer distribution:\n{train_df['answer'].value_counts()}")

# Split data
train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['answer']
)

print(f"\nTrain: {len(train_data)}, Validation: {len(val_data)}")

✅ Loaded 2000 training samples
✅ Loaded 500 test samples

Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Train: 1600, Validation: 400


### 10. Train Models

In [11]:
# Train Scratch Model
scratch_preprocessor = AdvancedDataPreprocessor()
X_scratch, y_scratch, _ = scratch_preprocessor.load_data()

scratch_best_state, scratch_cv_score = train_scratch_model(X_scratch, y_scratch, scratch_preprocessor)

scratch_model = EnhancedScratchModel(
    vocab_size=scratch_preprocessor.vocab_size,
    embedding_dim=config.scratch_embedding_dim,
    hidden_dim=config.scratch_hidden_dim,
    num_layers=config.scratch_num_layers,
    max_length=config.scratch_max_sequence_length,
    dropout=config.scratch_dropout
).to(DEVICE)

if scratch_best_state:
    scratch_model.load_state_dict(scratch_best_state)

print(f"\nScratch Model - CV Score: {scratch_cv_score:.4f}")

# Train BERT Model
print("\n" + "="*60)
print("Training BERT Model")
print("="*60)

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
if bert_tokenizer.pad_token is None:
    bert_tokenizer.pad_token = bert_tokenizer.eos_token

train_dataset = prepare_data(train_data, bert_tokenizer, config.max_length, is_train=True, augment=True)
val_dataset = prepare_data(val_data, bert_tokenizer, config.max_length, is_train=True)

train_loader = DataLoader(train_dataset, batch_size=config.train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.eval_batch_size, shuffle=False)

bert_model = PretrainedMCQModel('bert-base-uncased', dropout=0.3)
bert_model, bert_acc, bert_f1 = train_pretrained_model(
    bert_model, train_loader, val_loader, config, 'bert'
)

print(f"\nBERT Model - Best Val Acc: {bert_acc:.4f}, F1: {bert_f1:.4f}")

# Clean up
del train_loader, val_loader
gc.collect()
torch.cuda.empty_cache()

# Train DistilBERT Model
print("\n" + "="*60)
print("Training DistilBERT Model")
print("="*60)

distil_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
if distil_tokenizer.pad_token is None:
    distil_tokenizer.pad_token = distil_tokenizer.eos_token

train_dataset = prepare_data(train_data, distil_tokenizer, config.max_length, is_train=True, augment=True)
val_dataset = prepare_data(val_data, distil_tokenizer, config.max_length, is_train=True)

train_loader = DataLoader(train_dataset, batch_size=config.train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.eval_batch_size, shuffle=False)

distil_model = PretrainedMCQModel('distilbert-base-uncased', dropout=0.3)
distil_model, distil_acc, distil_f1 = train_pretrained_model(
    distil_model, train_loader, val_loader, config, 'distilbert'
)

print(f"\nDistilBERT Model - Best Val Acc: {distil_acc:.4f}, F1: {distil_f1:.4f}")

# Clean up
del train_loader, val_loader
gc.collect()
torch.cuda.empty_cache()


Loading data...
Loaded 2000 samples
Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Cleaning text...

Building vocabulary...
Vocabulary size: 3744

Training Scratch Model with Cross-Validation

Fold 1/3
Epoch  1/15 | Train Loss: 1.2647 | Train Acc: 0.5364 | Val Acc: 0.4003 | Val F1: 0.3708
Epoch  2/15 | Train Loss: 0.8269 | Train Acc: 0.8042 | Val Acc: 0.8951 | Val F1: 0.9006
Epoch  3/15 | Train Loss: 0.7134 | Train Acc: 0.8545 | Val Acc: 0.6567 | Val F1: 0.5813
Epoch  4/15 | Train Loss: 0.7461 | Train Acc: 0.8665 | Val Acc: 0.9250 | Val F1: 0.9238
Epoch  5/15 | Train Loss: 0.6421 | Train Acc: 0.8845 | Val Acc: 0.5562 | Val F1: 0.5756
Epoch  6/15 | Train Loss: 0.6148 | Train Acc: 0.9055 | Val Acc: 0.9430 | Val F1: 0.9451
Epoch  7/15 | Train Loss: 0.5797 | Train Acc: 0.9205 | Val Acc: 0.9340 | Val F1: 0.9347
Epoch  8/15 | Train Loss: 0.5324 | Train Acc: 0.9430 | Val Acc: 0.9235 | Val F1: 0.9254
Epoch  9/15 | Train Loss: 0.4798 | Train 

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 2: Train Acc=0.7481, Val Acc=0.8000, Val F1=0.7111


Epoch 3 Val: 100%|██████████| 63/63 [00:15<00:00,  4.09it/s]


Epoch 3: Train Acc=0.7416, Val Acc=0.8000, Val F1=0.7111


Epoch 4 Val: 100%|██████████| 63/63 [00:15<00:00,  4.08it/s]


Epoch 4: Train Acc=0.7574, Val Acc=0.8000, Val F1=0.7111

DistilBERT Model - Best Val Acc: 0.8000, F1: 0.7111


### 11. Ensemble Predictions

In [12]:
def predict_scratch(model, test_df, preprocessor, config):
    """Generate predictions using scratch model."""
    model.eval()
    
    test_df['clean_prompt'] = test_df['prompt'].apply(preprocessor.clean_text)
    for col in config.option_columns:
        if col in test_df.columns:
            test_df[f'clean_{col}'] = test_df[col].apply(preprocessor.clean_text)
    
    test_df['combined_text'] = test_df.apply(
        lambda row: preprocessor.create_enhanced_text(
            row['clean_prompt'],
            [row.get(f'clean_{col}', '') for col in config.option_columns]
        ), axis=1
    )
    
    test_sequences = preprocessor.texts_to_sequences(test_df['combined_text'].tolist())
    test_tensor = torch.tensor(test_sequences, dtype=torch.long)
    
    all_scores = []
    with torch.no_grad():
        for i in range(0, len(test_tensor), config.train_batch_size):
            batch = test_tensor[i:i+config.train_batch_size].to(DEVICE)
            outputs = model(batch)
            probs = F.softmax(outputs, dim=1)
            all_scores.extend(probs.cpu().numpy())
    
    return np.array(all_scores)

def predict_pretrained(model, test_df, tokenizer, config, use_tta=True):
    """Generate predictions using pretrained model with TTA."""
    model.to(DEVICE)
    model.eval()
    
    option_cols = ['A', 'B', 'C', 'D', 'E'] if 'E' in test_df.columns else ['A', 'B', 'C', 'D']
    all_scores = []
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
        question = str(row['prompt'])
        options = [str(row[col]) for col in option_cols]
        
        scores = []
        for opt in options:
            # TTA: Multiple augmented versions
            if use_tta:
                texts = [
                    f"Question: {question} Option: {opt}",
                    f"{question} {opt}",
                    f"Q: {question} A: {opt}"
                ]
            else:
                texts = [f"Question: {question} Option: {opt}"]
            
            opt_scores = []
            for text in texts:
                encoding = tokenizer(
                    text,
                    truncation=True,
                    padding='max_length',
                    max_length=config.max_length,
                    return_tensors='pt'
                )
                
                input_ids = encoding['input_ids'].to(DEVICE)
                attention_mask = encoding['attention_mask'].to(DEVICE)
                
                with torch.no_grad():
                    outputs = model(input_ids, attention_mask)
                    probs = F.softmax(outputs['logits'], dim=-1)
                    opt_scores.append(probs[0, 1].item())
            
            scores.append(np.mean(opt_scores))
        
        all_scores.append(scores)
    
    return np.array(all_scores)

def ensemble_predict(models_dict, test_df, tokenizers, config):
    """Ensemble predictions from multiple models."""
    all_preds = []
    weights = []
    
    for name, data in models_dict.items():
        print(f"\nPredicting with {name}...")
        if name == 'scratch':
            preds = predict_scratch(data['model'], test_df, data['preprocessor'], config)
        else:
            preds = predict_pretrained(data['model'], test_df, tokenizers[name], config, use_tta=True)
        all_preds.append(preds)
        weights.append(data['weight'])
    
    weights = np.array(weights).reshape(-1, 1, 1)
    ensemble = np.average(np.array(all_preds), axis=0, weights=weights.flatten())
    return ensemble

# Prepare models
models_dict = {
    'scratch': {
        'model': scratch_model, 
        'weight': config.ensemble_weights[0],
        'preprocessor': scratch_preprocessor
    },
    'bert': {
        'model': bert_model, 
        'weight': config.ensemble_weights[1]
    },
    'distilbert': {
        'model': distil_model, 
        'weight': config.ensemble_weights[2]
    }
}

tokenizers = {
    'bert': bert_tokenizer,
    'distilbert': distil_tokenizer
}

# Generate ensemble predictions
print("\n" + "="*60)
print("Generating Ensemble Predictions with TTA")
print("="*60)

ensemble_scores = ensemble_predict(models_dict, test_df, tokenizers, config)



Generating Ensemble Predictions with TTA

Predicting with scratch...

Predicting with bert...


Predicting: 100%|██████████| 500/500 [02:10<00:00,  3.83it/s]



Predicting with distilbert...


Predicting: 100%|██████████| 500/500 [01:08<00:00,  7.33it/s]


### 12. Generate Submission

In [13]:
def create_submission(scores, test_df):
    """Create submission file for Kaggle."""
    submission = []
    for idx, row in test_df.iterrows():
        top_idx = np.argsort(scores[idx])[::-1][:3]
        pred = ' '.join([chr(ord('A') + i) for i in top_idx])
        submission.append({'ID': row['id'], 'Prediction': pred})
    return pd.DataFrame(submission)

submission = create_submission(ensemble_scores, test_df)
submission.to_csv('submission.csv', index=False)

print(f"\n Submission saved! Shape: {submission.shape}")
print(submission.head())


 Submission saved! Shape: (500, 2)
   ID Prediction
0   1      A D B
1   2      B E D
2   3      B E C
3   4      E C B
4   5      C E D


### 13. Summary

In [14]:
print("\n" + "="*60)
print("PROJECT SUMMARY")
print("="*60)

print("\nModel Performance:")
print("-" * 40)
print(f"Scratch Model     - CV Score: {scratch_cv_score:.4f}")
print(f"BERT Model        - Accuracy: {bert_acc:.4f}, F1: {bert_f1:.4f}")
print(f"DistilBERT Model  - Accuracy: {distil_acc:.4f}, F1: {distil_f1:.4f}")

print("\n Ensemble Weights:")
for name, data in models_dict.items():
    print(f"  {name}: {data['weight']:.2f}")

print(f"\n Submission: {len(submission)} rows")

if wandb_enabled:
    wandb.finish()

print("\n" + "="*60)
print("Comleted")
print("="*60)

torch.cuda.empty_cache()
print("\n Cache cleared!")

wandb: updating run metadata



PROJECT SUMMARY

Model Performance:
----------------------------------------
Scratch Model     - CV Score: 0.9940
BERT Model        - Accuracy: 0.7895, F1: 0.7136
DistilBERT Model  - Accuracy: 0.8000, F1: 0.7111

 Ensemble Weights:
  scratch: 0.10
  bert: 0.45
  distilbert: 0.45

 Submission: 500 rows


wandb: uploading summary, console lines 188-212
wandb: 
wandb: Run history:
wandb:        bert_train_acc ▅█▁▃
wandb:         bert_train_f1 ▅█▁▃
wandb:       bert_train_loss ▃▁█▃
wandb:          bert_val_acc ▁▁▁▁
wandb:           bert_val_f1 ▁▁▁▁
wandb:         bert_val_loss ▁▁▁▁
wandb:  distilbert_train_acc ▂▄▁█
wandb:   distilbert_train_f1 ▁▄▁█
wandb: distilbert_train_loss ▇█▇▁
wandb:    distilbert_val_acc ▁▁▁▁
wandb:                    +4 ...
wandb: 
wandb: Run summary:
wandb:        bert_train_acc 0.644
wandb:         bert_train_f1 0.65974
wandb:       bert_train_loss 0.6386
wandb:          bert_val_acc 0.7895
wandb:           bert_val_f1 0.71362
wandb:         bert_val_loss 0.6428
wandb:  distilbert_train_acc 0.75738
wandb:   distilbert_train_f1 0.7128
wandb: distilbert_train_loss 0.63903
wandb:    distilbert_val_acc 0.8
wandb:                    +4 ...
wandb: 
wandb: 🚀 View run mcq_training_1784671104 at: https://wandb.ai/23f3000080-dl-genai-project/23f3000080-t22026/runs/i4kskway


Comleted

 Cache cleared!
